In [ ]:
class PINNBranch(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        # Beta head: estimasi beta_R, beta_G, beta_B
        self.beta_head = nn.Sequential(
            nn.Linear(channels, 64), nn.GELU(),
            nn.Linear(64, 3),  nn.Softplus()  # nilai positif
        )
        # Depth head: estimasi kedalaman ternormalisasi
        self.depth_head = nn.Sequential(
            nn.Linear(channels, 64), nn.GELU(),
            nn.Linear(64, 1),  nn.Sigmoid()   # range (0,1)
        )
    def forward(self, feat):
        v = self.gap(feat).flatten(1)
        beta = self.beta_head(v)   # (B, 3)
        d    = self.depth_head(v)  # (B, 1)
        return beta, d

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': nafnet.parameters(), 'lr': 3e-4},
    {'params': pinn.parameters(),  'lr': 1e-3},
], weight_decay=1e-4)
 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=60, eta_min=1e-5)
 
scaler = torch.cuda.amp.GradScaler()  # Mixed Precision
 
# Fase 1: Soft-Freeze Warmup (epoch 1-2)
for param in nafnet.encoder_stages[:2].parameters():
    param.requires_grad = False
 
# Fase 2: Full Fine-tuning (epoch 3-60)
for param in nafnet.parameters():
    param.requires_grad = True
